# บทที่ 10 โครงข่ายประสาทเทียมและการเรียนรู้ที่สอดคล้องกับฟิสิกส์
โน้ตบุ๊กประกอบตำรา *วิทยาศาสตร์ข้อมูลและการเรียนรู้ของเครื่องสำหรับนักฟิสิกส์*
โค้ดทุกเซลล์คัดลอกมาจากสคริปต์ในโฟลเดอร์ `ch10_neural_networks/` หัวเซลล์ **โค้ดที่ N.M** ตรงกับเลขในตำรา

**วิธีใช้**
1. รันเซลล์ **เตรียมโค้ด** ด้านล่างก่อนเสมอ (ทุกครั้งที่เปิดโน้ตบุ๊กหรือเริ่ม session ใหม่)
2. แต่ละหัวข้อเริ่มด้วยเซลล์ที่ขึ้นต้นด้วย `%reset -f` ซึ่งล้างตัวแปรเก่า ทำให้ผลเหมือนรันสคริปต์นั้นเดี่ยว ๆ
   ให้รันเซลล์ในหัวข้อเดียวกันเรียงจากบนลงล่าง (Shift+Enter) หรือใช้ Runtime → Run all
3. ใช้หน่วยประมวลผลกลาง (CPU) ตามค่าเริ่มต้น ไม่ต้องเปลี่ยนเป็น GPU
4. ไฟล์ที่สร้างขึ้นอยู่ใน `ch10_neural_networks/outputs/` และหายเมื่อ session จบ ถ้าต้องการเก็บ ให้คัดลอกไปยัง Google Drive

ตัวเลขส่วนใหญ่ควรตรงกับตำรา การคำนวณที่ไวต่อการปัดเศษ (การฝึกโครงข่ายประสาทเทียม
และมอนติคาร์โลในพิกัดต่อเนื่อง) อาจต่างเล็กน้อยเมื่อรันบนเครื่องอื่น ดู `ch10_neural_networks/README.md`


In [ ]:
# เตรียมโค้ด: ดึงโค้ดจาก GitHub (เมื่อรันบน Colab) และติดตั้งเฉพาะไลบรารีที่ยังไม่มี
import os, sys, subprocess, importlib.util
CHAPTER = 'ch10_neural_networks'
REPO = 'https://github.com/busarapa-stack/dsml-for-physicists.git'
if 'google.colab' in sys.modules:
    ROOT = '/content/dsml-for-physicists'
    if not os.path.isdir(ROOT):
        subprocess.run(['git', 'clone', '-q', REPO, ROOT], check=True)
else:                                   # Jupyter บนเครื่องตนเอง: เปิดจากโฟลเดอร์ notebooks/
    ROOT = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
os.chdir(os.path.join(ROOT, CHAPTER))
os.makedirs('data', exist_ok=True); os.makedirs('outputs', exist_ok=True)
NEED = {'matplotlib': 'matplotlib', 'numpy': 'numpy', 'scipy': 'scipy', 'sklearn': 'scikit-learn>=1.3', 'torch': 'torch>=2.0'}
missing = [req for mod, req in NEED.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
print('โฟลเดอร์ทำงาน:', os.getcwd())
print('ติดตั้งเพิ่ม:', ', '.join(missing) if missing else 'ไม่มี')


## `01_higgs_mlp.py` — โค้ดที่ 10.1, 10.2

Ch.10  A two-layer network (MLP) for the Chapter 6 toy Higgs data, trained with PyTorch.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 01_higgs_mlp.py  จากโฟลเดอร์ของบท
__file__ = '01_higgs_mlp.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.10  A two-layer network (MLP) for the Chapter 6 toy Higgs data, trained with PyTorch.

Book references: ssec:pytorch-mlp, ssec:train-loop, code:mlp and code:train (line for line)
Numbers quoted in the text: 4,610 parameters; AUC 0.979, accuracy 0.935, final loss 0.126;
logistic regression 0.703, random forest 0.975 (Chapter 6); without activations AUC 0.688.
"""
import os
import sys
from pathlib import Path

import numpy as np

HERE = Path(__file__).resolve().parent
sys.path.insert(0, str(HERE))
from _shared import higgs_split, section, CODE

os.chdir(HERE)
Path('outputs').mkdir(exist_ok=True)
nb = higgs_split({})
X_train, X_val, y_train, y_val = nb['X_train'], nb['X_val'], nb['y_train'], nb['y_val']

### โค้ดที่ 10.1

In [ ]:
# --- code:mlp ----------------------------------------------------------------
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler

class HiggsMLP(nn.Module):
    def __init__(self, n_in=4, hidden=64, n_classes=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, n_classes))      # raw scores (logits), no softmax
    def forward(self, x):
        return self.net(x)

scaler = StandardScaler().fit(X_train)         # X_train, X_val from Chapter 6
Xt = torch.tensor(scaler.transform(X_train), dtype=torch.float32)
yt = torch.tensor(y_train)
Xv = torch.tensor(scaler.transform(X_val), dtype=torch.float32)

torch.manual_seed(0)
model = HiggsMLP()
print(sum(p.numel() for p in model.parameters()), "parameters")

### โค้ดที่ 10.2

In [ ]:
# --- code:train --------------------------------------------------------------
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score

loader = DataLoader(TensorDataset(Xt, yt), batch_size=64, shuffle=True)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()
history = []
for epoch in range(30):
    model.train()                                # training mode
    total = 0.0
    for xb, yb in loader:
        loss = loss_fn(model(xb), yb)            # 1. forward pass and loss
        optimizer.zero_grad()                    # 2. clear old gradients
        loss.backward()                          # 3. backpropagation
        optimizer.step()                         # 4. update parameters
        total += loss.item() * len(xb)
    history.append(total / len(Xt))

model.eval()                                     # evaluation mode
with torch.no_grad():                            # no gradient bookkeeping
    p_val = torch.softmax(model(Xv), dim=1)[:, 1].numpy()
print("AUC =", round(roc_auc_score(y_val, p_val), 4))

**ส่วนตรวจผล** (ไม่อยู่ในตำรา): พิมพ์ตัวเลขที่ตำรายกมาเพื่อเทียบ

In [ ]:
# --- end of listings ---------------------------------------------------------

if __name__ == '__main__':
    acc = np.mean((p_val > 0.5) == y_val)
    print(f"final training loss {history[-1]:.3f} (epoch 1: {history[0]:.3f}); accuracy at P > 0.5: {acc:.3f}")

    # the same network with the two ReLUs removed: a product of linear maps is linear
    torch.manual_seed(0)
    linear = nn.Sequential(nn.Linear(4, 64), nn.Linear(64, 64), nn.Linear(64, 2))
    opt = torch.optim.Adam(linear.parameters(), lr=1e-3)
    for epoch in range(30):
        linear.train()
        for xb, yb in loader:
            loss = loss_fn(linear(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
    linear.eval()
    with torch.no_grad():
        p_lin = torch.softmax(linear(Xv), dim=1)[:, 1].numpy()
    print(f"no activation functions: AUC = {roc_auc_score(y_val, p_lin):.3f}")

    # Chapter 6 baselines on the same split (code:lr and code:rf)
    ch6 = dict(nb)
    ch6['__file__'] = str(CODE / 'ch06_classification' / '02_logreg_forest.py')
    import contextlib, io
    with contextlib.redirect_stdout(io.StringIO()):          # code:lr prints its coefficients
        exec(section(ch6['__file__'], '\n# --- code:lr', '\n# --- end of listings'), ch6)
    for name in ('lr_pipe', 'rf_pipe'):
        print(f"Chapter 6 {name}: AUC = {roc_auc_score(y_val, ch6[name].predict_proba(X_val)[:, 1]):.3f}")

    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.plot(range(1, 31), history, 'o-', ms=3)
    ax.set_xlabel('epoch'); ax.set_ylabel('training loss (cross entropy)')
    fig.tight_layout(); fig.savefig('outputs/ch10_higgs_loss.png', dpi=100)
    print("saved outputs/ch10_higgs_loss.png")

In [ ]:
# แสดงรูปที่ส่วนนี้สร้าง (สคริปต์บันทึกรูปลงไฟล์ใน outputs/)
from IPython.display import Image, display
import glob, os
for _f in sorted(glob.glob('outputs/**/*.png', recursive=True)):
    if os.path.getmtime(_f) >= float(os.environ['NB_T0']):
        print(_f); display(Image(_f))


## `02_learning_rate_scan.py` — โค้ดที่ 10.2

Ch.10  Learning-rate scan for the Higgs MLP and the 'dead ReLU' network at lr = 1.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 02_learning_rate_scan.py  จากโฟลเดอร์ของบท
__file__ = '02_learning_rate_scan.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.10  Learning-rate scan for the Higgs MLP and the 'dead ReLU' network at lr = 1.

Book references: ssec:loss-thermo, tab:lr-scan and the paragraph after it
The loop is code:train with only the learning rate changed (same seed, same data order).
Numbers quoted in the text (final training loss, validation AUC):
  1e-5: 0.327, 0.905   1e-4: 0.139, 0.977   1e-3: 0.126, 0.979
  1e-2: 0.129, 0.979   1e-1: 0.182, 0.966   1:    0.520, 0.500
  at lr = 1 all 64 units of the second hidden layer output 0 for every event; P ~ 0.22 for every event
About 2 minutes.
"""
import os
import sys
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, TensorDataset

HERE = Path(__file__).resolve().parent
sys.path.insert(0, str(HERE))
from _shared import higgs_split, listing

os.chdir(HERE)
nb = higgs_split({})
listing(nb, '01_higgs_mlp.py', 'mlp', '\n# --- code:train')      # HiggsMLP, Xt, yt, Xv
HiggsMLP, Xt, yt, Xv, y_val = nb['HiggsMLP'], nb['Xt'], nb['yt'], nb['Xv'], nb['y_val']


def fit(lr, epochs=30, seed=0):
    torch.manual_seed(seed)
    model = HiggsMLP()
    loader = DataLoader(TensorDataset(Xt, yt), batch_size=64, shuffle=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    history = []
    for epoch in range(epochs):
        model.train()
        total = 0.0
        for xb, yb in loader:
            loss = loss_fn(model(xb), yb)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            total += loss.item() * len(xb)
        history.append(total / len(Xt))
    model.eval()
    with torch.no_grad():
        p = torch.softmax(model(Xv), dim=1)[:, 1].numpy()
    return model, history, p


print(f"{'lr':>7s}  loss(epoch 1, 5, 10, 30)          AUC")
for lr in (1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0):
    model, h, p = fit(lr)
    auc = roc_auc_score(y_val, p) if p.std() > 1e-6 else 0.5    # constant scores -> AUC 0.5
    print(f"{lr:7.0e}  {h[0]:.3f} {h[4]:.3f} {h[9]:.3f} {h[-1]:.3f}      {auc:.3f}")

# the network trained with lr = 1: which hidden units ever fire?
net = model.net
with torch.no_grad():
    h1 = torch.relu(net[0](Xt))
    h2 = torch.relu(net[2](h1))
    p_all = torch.softmax(model(Xt), dim=1)[:, 1]
print(f"lr = 1: units active for at least one training event: layer 1 {(h1.max(0).values > 0).sum().item()}/64, "
      f"layer 2 {(h2.max(0).values > 0).sum().item()}/64; P(signal) {p_all.min():.3f}-{p_all.max():.3f} for every event")
print(f"loss of a constant prediction 0.2: {-(0.8 * np.log(0.8) + 0.2 * np.log(0.2)):.3f}")

## `03_polymer_chain.py` — โค้ดที่ 10.3, 10.4

Ch.10  Lennard-Jones chain (N = 30): Monte Carlo ensemble at 15 temperatures and physical indicators.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 03_polymer_chain.py  จากโฟลเดอร์ของบท
__file__ = '03_polymer_chain.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.10  Lennard-Jones chain (N = 30): Monte Carlo ensemble at 15 temperatures and physical indicators.

Book references: ssec:polymer-data, ssec:polymer-fe, code:chain-gen and code:chain-features (line for line)
Writes data/chain.npz (conf, ener, T_lab, T_grid), reused by 04, 06 and exercise E10.6.
Numbers quoted in the text: generation about two minutes; <Rg^2> 2.6 (T = 0.6) -> 8.6 (T = 3.4);
tanh fit of <Rg^2>(T): T* = 1.90 +/- 0.04, w = 1.0; d<Rg^2>/dT 2.6-3.1 for T = 1.4-2.2, no sharp peak;
C_v/N decreasing with a weak shoulder, large at T = 0.6.
"""
import os
import time
from pathlib import Path

HERE = Path(__file__).resolve().parent
os.chdir(HERE)
Path('data').mkdir(exist_ok=True)
Path('outputs').mkdir(exist_ok=True)
_t0 = time.time()

### โค้ดที่ 10.3

In [ ]:
# --- code:chain-gen ----------------------------------------------------------
import numpy as np

def make_chain_ensemble(N=30, T=2.0, n_samples=400, sweeps_between=10,
                        n_equil=2000, seed=0):
    """Flexible chain: bond length 1, Lennard-Jones (eps = sigma = 1) between
    beads with |i - j| >= 2. Metropolis sampling with pivot and crankshaft moves.
    Returns centred configurations (n_samples, N, 3) and energies."""
    rng = np.random.default_rng(seed)
    iu, ju = np.triu_indices(N, k=2)
    def energy(r):
        s6 = 1.0 / np.sum((r[iu] - r[ju])**2, axis=1)**3
        return np.sum(4.0 * (s6 * s6 - s6))
    def rotation(axis, angle):                         # Rodrigues' formula
        a = axis / np.linalg.norm(axis)
        K = np.array([[0, -a[2], a[1]], [a[2], 0, -a[0]], [-a[1], a[0], 0]])
        return np.eye(3) + np.sin(angle) * K + (1 - np.cos(angle)) * K @ K
    r = np.zeros((N, 3)); r[:, 0] = np.arange(N)       # straight initial chain
    E, beta = energy(r), 1.0 / T
    confs, energies = [], []
    for m in range((n_equil + n_samples * sweeps_between) * N):
        new = r.copy()
        if rng.random() < 0.5:                         # pivot move
            k = rng.integers(1, N - 1)
            R = rotation(rng.normal(size=3), rng.uniform(-np.pi, np.pi) * 0.5)
            if rng.random() < 0.5:
                new[k+1:] = (r[k+1:] - r[k]) @ R.T + r[k]
            else:
                new[:k] = (r[:k] - r[k]) @ R.T + r[k]
        else:                                          # crankshaft or end move
            i = rng.integers(0, N)
            if 0 < i < N - 1:
                R = rotation(r[i+1] - r[i-1], rng.uniform(-np.pi, np.pi))
                new[i] = (r[i] - r[i-1]) @ R.T + r[i-1]
            else:
                j = 1 if i == 0 else N - 2
                v = rng.normal(size=3)
                new[i] = r[j] + v / np.linalg.norm(v)
        E_new = energy(new)
        if E_new <= E or rng.random() < np.exp(-beta * (E_new - E)):
            r, E = new, E_new
        sweep, rest = divmod(m + 1, N)
        if rest == 0 and sweep > n_equil and (sweep - n_equil) % sweeps_between == 0:
            confs.append(r - r.mean(axis=0))
            energies.append(E)
    return np.array(confs), np.array(energies)

T_grid = np.round(np.arange(0.6, 3.41, 0.2), 1)       # 15 temperatures
data = [make_chain_ensemble(T=T, seed=100 + k) for k, T in enumerate(T_grid)]
conf = np.concatenate([d[0] for d in data])            # (6000, 30, 3)
ener = np.concatenate([d[1] for d in data])
T_lab = np.repeat(T_grid, 400)

### โค้ดที่ 10.4

In [ ]:
# --- code:chain-features -----------------------------------------------------
N = conf.shape[1]
iu, ju = np.triu_indices(N, k=1)
dist = np.linalg.norm(conf[:, iu] - conf[:, ju], axis=2)   # 435 pair distances
rg2 = np.mean(np.sum(conf**2, axis=2), axis=1)              # squared radius of gyration
for T in T_grid:
    m = T_lab == T
    dRg2_dT = np.cov(rg2[m], ener[m])[0, 1] / T**2           # fluctuation formula
    print(f"T = {T:.1f}  <Rg^2> = {rg2[m].mean():5.2f}  "
          f"C_v/N = {ener[m].var() / T**2 / N:5.3f}  d<Rg^2>/dT = {dRg2_dT:4.2f}")

**ส่วนตรวจผล** (ไม่อยู่ในตำรา): พิมพ์ตัวเลขที่ตำรายกมาเพื่อเทียบ

In [ ]:
# --- end of listings ---------------------------------------------------------

if __name__ == '__main__':
    print(f"generation + features: {time.time() - _t0:.0f} s")
    np.savez('data/chain.npz', conf=conf, ener=ener, T_lab=T_lab, T_grid=T_grid)
    print("saved data/chain.npz", conf.shape)

    from scipy.optimize import curve_fit
    mean_rg2 = np.array([rg2[T_lab == T].mean() for T in T_grid])
    f = lambda T, a, b, Ts, w: a + b * np.tanh((T - Ts) / w)
    p, c = curve_fit(f, T_grid, mean_rg2, p0=[5, 3, 1.8, 0.5])
    e = np.sqrt(np.diag(c))
    print(f"tanh fit of <Rg^2>(T): T* = {p[2]:.2f} +/- {e[2]:.2f}, w = {p[3]:.2f} +/- {e[3]:.2f}")

    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.plot(T_grid, mean_rg2, 'o', label=r'$\langle R_g^2\rangle$')
    Ts = np.linspace(0.6, 3.4, 200)
    ax.plot(Ts, f(Ts, *p), '-', label=f'tanh fit, T* = {p[2]:.2f}')
    ax.set_xlabel('T'); ax.set_ylabel(r'$\langle R_g^2\rangle$'); ax.legend()
    fig.tight_layout(); fig.savefig('outputs/ch10_chain_rg2.png', dpi=100)
    print("saved outputs/ch10_chain_rg2.png")

In [ ]:
# แสดงรูปที่ส่วนนี้สร้าง (สคริปต์บันทึกรูปลงไฟล์ใน outputs/)
from IPython.display import Image, display
import glob, os
for _f in sorted(glob.glob('outputs/**/*.png', recursive=True)):
    if os.path.getmtime(_f) >= float(os.environ['NB_T0']):
        print(_f); display(Image(_f))


## `04_polymer_classifier.py` — โค้ดที่ 10.5

Ch.10  Learning by extremes (Carrasquilla and Melko) on the polymer chain: where does P(globule) cross 0.5?

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 04_polymer_classifier.py  จากโฟลเดอร์ของบท
__file__ = '04_polymer_classifier.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.10  Learning by extremes (Carrasquilla and Melko) on the polymer chain: where does P(globule) cross 0.5?

Book references: ssec:polymer-nn-train, ssec:nn-order-param, code:polymer-nn (line for line), tab:polymer-nn
Needs data/chain.npz from 03_polymer_chain.py (generated here if missing, about two minutes).
Numbers quoted in the text:
  validation accuracy 0.994; mean P = 1.00 for T <= 1.0, 0.70 / 0.40 / 0.29 at T = 1.2 / 1.4 / 1.6, 0 for T >= 3.2;
  T*_NN = 1.33; within-T SD of P 0.40-0.46 for T = 1.2-1.8; 86-90 % of configurations have P < 0.1 or P > 0.9
  tab:polymer-nn  (ends <=0.8/>=3.2, <=1.0/>=3.0, <=1.2/>=2.8)
     pair distances 1.25 / 1.33 / 1.43;  Rg^2 alone 1.33 / 1.55 / 1.56;  raw coordinates 0.93 / 1.32 / 1.41
  Rg^2 alone separates the training ends 99 %; raw coordinates validate 0.94-0.99 vs 0.98-0.99;
  raw coordinates with ends <= 0.8: mean P 0.21 at T = 1.0 but 0.41 at T = 1.2
About 1.5 minutes.
"""
import os
import sys
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn

HERE = Path(__file__).resolve().parent
sys.path.insert(0, str(HERE))
from _shared import chain_data

os.chdir(HERE)
_d = chain_data()
conf, ener, T_lab, T_grid, dist, rg2 = (_d[k] for k in ('conf', 'ener', 'T_lab', 'T_grid', 'dist', 'rg2'))

### โค้ดที่ 10.5

In [ ]:
# --- code:polymer-nn ---------------------------------------------------------
def train_phase_classifier(F, T, T_low, T_high, seed=0, hidden=64, epochs=60):
    """Learning by extremes: label T <= T_low as globule (1), T >= T_high as coil (0)."""
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    train = (T <= T_low) | (T >= T_high)
    y = (T <= T_low).astype(np.int64)
    mu, sd = F[train].mean(axis=0), F[train].std(axis=0) + 1e-9
    X_all = torch.tensor((F - mu) / sd, dtype=torch.float32)
    y_all = torch.tensor(y)
    idx = rng.permutation(np.where(train)[0])
    i_tr, i_va = idx[:int(0.8 * len(idx))], idx[int(0.8 * len(idx)):]
    model = nn.Sequential(nn.Linear(F.shape[1], hidden), nn.ReLU(), nn.Dropout(0.2),
                          nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(0.2),
                          nn.Linear(hidden, 2))
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    loss_fn = nn.CrossEntropyLoss()
    for epoch in range(epochs):
        model.train()
        perm = rng.permutation(i_tr)
        for b in range(0, len(perm), 64):
            bi = perm[b:b + 64]
            loss = loss_fn(model(X_all[bi]), y_all[bi])
            opt.zero_grad(); loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():
        P = torch.softmax(model(X_all), dim=1)[:, 1].numpy()      # P(globule)
    val_acc = np.mean((P[i_va] > 0.5) == y[i_va])
    return model, X_all, P, val_acc

def crossing(P, T, T_grid):
    """Temperature where the mean P(globule) crosses 0.5 (linear interpolation)."""
    Pm = np.array([P[T == t].mean() for t in T_grid])
    k = np.where((Pm[:-1] - 0.5) * (Pm[1:] - 0.5) <= 0)[0][0]
    return T_grid[k] + (0.5 - Pm[k]) * (T_grid[k+1] - T_grid[k]) / (Pm[k+1] - Pm[k]), Pm

model, X_all, P, acc = train_phase_classifier(dist, T_lab, T_low=1.0, T_high=3.0)
T_star, Pm = crossing(P, T_lab, T_grid)
print("validation accuracy:", round(acc, 3), " T* =", round(T_star, 2))
print("mean P(globule):", Pm.round(2))
print("spread of P within each T:", np.array([P[T_lab == t].std() for t in T_grid]).round(2))

**ส่วนตรวจผล** (ไม่อยู่ในตำรา): พิมพ์ตัวเลขที่ตำรายกมาเพื่อเทียบ

In [ ]:
# --- end of listings ---------------------------------------------------------

if __name__ == '__main__':
    sd = np.array([P[T_lab == t].std() for t in T_grid])
    sure = np.array([np.mean((P[T_lab == t] < 0.1) | (P[T_lab == t] > 0.9)) for t in T_grid])
    band = (T_grid >= 1.2) & (T_grid <= 1.8)
    print(f"T = 1.2-1.8: within-T SD of P {sd[band].min():.2f}-{sd[band].max():.2f}; "
          f"fraction with P < 0.1 or P > 0.9: {sure[band].min():.2f}-{sure[band].max():.2f}")

    print("\ntab:polymer-nn  crossing of mean P = 0.5 (validation accuracy)")
    features = {'pair distances (435)': dist, 'Rg^2 alone': rg2[:, None],
                'raw coordinates (90)': conf.reshape(len(conf), -1)}
    ends = [(0.8, 3.2), (1.0, 3.0), (1.2, 2.8)]
    print(f"{'features':22s}" + ''.join(f"   <= {lo}, >= {hi}" for lo, hi in ends))
    for name, F in features.items():
        row = []
        for lo, hi in ends:
            _, _, Pq, a = train_phase_classifier(F, T_lab, lo, hi)
            ts, pm = crossing(Pq, T_lab, T_grid)
            row.append(f"   {ts:5.2f} ({a:.3f})  ")
            if name.startswith('raw') and lo == 0.8:
                raw_pm = pm
        print(f"{name:22s}" + ''.join(row))
    print(f"raw coordinates, ends <= 0.8 / >= 3.2: mean P at T = 1.0 {raw_pm[T_grid == 1.0][0]:.2f}, "
          f"at T = 1.2 {raw_pm[T_grid == 1.2][0]:.2f}")
    print(f"midpoint of <Rg^2>(T) from the tanh fit (03_polymer_chain.py): 1.90 +/- 0.04")

## `05_ising_configs.py` — โค้ดที่ 10.6

Ch.10  16x16 Ising configurations for exercise E10.4 (code:ising-configs, line for line).

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 05_ising_configs.py  จากโฟลเดอร์ของบท
__file__ = '05_ising_configs.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.10  16x16 Ising configurations for exercise E10.4 (code:ising-configs, line for line).

Uses init_lattice and metropolis_step from the Chapter 4 listing code:metropolis.
Writes data/ising.npz (spins (3400, 256), T_spin), used by exercises/E10_4_ising_extremes.py.
Numbers quoted in the text: the run takes about one minute; <|m|> ~ 0.58 at T = 2.4.
"""
import os
import sys
import time
from pathlib import Path

import numpy as np

HERE = Path(__file__).resolve().parent
sys.path.insert(0, str(HERE))
from _shared import ch4_metropolis

os.chdir(HERE)
Path('data').mkdir(exist_ok=True)
ch4_metropolis(globals())      # init_lattice, metropolis_step share this module's rng, as in the book
_t0 = time.time()

### โค้ดที่ 10.6

In [ ]:
# --- code:ising-configs ------------------------------------------------------
# uses init_lattice and metropolis_step from the Chapter 4 listing
def ising_configs(L, T, n_conf, sweeps_between=10, n_thermalize=500):
    lattice = init_lattice(L)
    beta = 1.0 / T
    for _ in range(n_thermalize):
        metropolis_step(lattice, beta)
    out = []
    for _ in range(n_conf):
        for _ in range(sweeps_between):
            metropolis_step(lattice, beta)
        out.append(lattice.copy())
    return np.array(out)

rng = np.random.default_rng(seed=2026)                 # restart the generator
T_ising = np.round(np.arange(1.6, 3.21, 0.1), 1)
spins = np.concatenate([ising_configs(16, T, 200) for T in T_ising])
spins = spins.reshape(len(spins), -1).astype(float)    # (3400, 256)
T_spin = np.repeat(T_ising, 200)

**ส่วนตรวจผล** (ไม่อยู่ในตำรา): พิมพ์ตัวเลขที่ตำรายกมาเพื่อเทียบ

In [ ]:
# --- end of listings ---------------------------------------------------------

if __name__ == '__main__':
    print(f"generated {spins.shape} in {time.time() - _t0:.0f} s")
    np.savez('data/ising.npz', spins=spins, T_spin=T_spin)
    print("saved data/ising.npz")
    m = spins.mean(axis=1)
    for T in T_ising:
        k = T_spin == T
        print(f"T = {T:.1f}  <|m|> = {np.abs(m[k]).mean():.2f}  <m> = {m[k].mean():+.2f}")

## `06_pinn_oscillator.py` — โค้ดที่ 10.7

Ch.10  Physics-informed neural network (PINN) for the damped oscillator vs a data-only network vs curve_fit.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 06_pinn_oscillator.py  จากโฟลเดอร์ของบท
__file__ = '06_pinn_oscillator.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.10  Physics-informed neural network (PINN) for the damped oscillator vs a data-only network vs curve_fit.

Book references: ssec:pinn-example, code:pinn (line for line); damped_oscillator from Chapter 5 (code:damped-cf)
Numbers quoted in the text: both trainings about two minutes; the network has ~8,500 parameters;
data only: RMSE 0.39 m (t <= 3) and 0.34 m (t > 3); PINN: 0.036 m and 0.111 m, gamma = 0.327, omega_0 = 6.020;
PINN error grows with distance from the data: 0.080 m for 3-4.5 s, 0.135 m for 4.5-6 s, where its amplitude
is ~0.07 m against 0.25 m (drifting toward the trivial solution x = 0);
curve_fit of A exp(-gamma t) cos(omega t + phi) on the same 30 points: gamma = 0.264 +/- 0.038,
omega = 6.038 +/- 0.037, RMSE 0.026 m and 0.031 m, in well under a second.
About 2 minutes.
"""
import os
import sys
import time
from pathlib import Path

import numpy as np

HERE = Path(__file__).resolve().parent
sys.path.insert(0, str(HERE))
from _shared import ch5_damped

os.chdir(HERE)
Path('outputs').mkdir(exist_ok=True)
damped_oscillator = ch5_damped({'__name__': 'ch05'})['damped_oscillator']
_t0 = time.time()

### โค้ดที่ 10.7

In [ ]:
# --- code:pinn ---------------------------------------------------------------
import torch
import torch.nn as nn

rng = np.random.default_rng(seed=2026)
t_obs = np.sort(rng.uniform(0, 3, 30))                 # data only in the first 3 s
x_obs = damped_oscillator(t_obs, 1.0, 0.3, 6.0, 0.5) + rng.normal(scale=0.05, size=30)
t_d = torch.tensor(t_obs, dtype=torch.float32)[:, None]
x_d = torch.tensor(x_obs, dtype=torch.float32)[:, None]
t_c = torch.linspace(0, 6, 300)[:, None].requires_grad_(True)   # collocation points

def train(use_physics, n_iter=30000, lam=1e-3, seed=0):
    torch.manual_seed(seed)
    net = nn.Sequential(nn.Linear(1, 64), nn.Tanh(), nn.Linear(64, 64), nn.Tanh(),
                        nn.Linear(64, 64), nn.Tanh(), nn.Linear(64, 1))
    log_g = torch.tensor(np.log(0.5), requires_grad=True)     # unknown gamma
    log_w2 = torch.tensor(np.log(30.0), requires_grad=True)   # unknown omega_0^2
    params = list(net.parameters()) + ([log_g, log_w2] if use_physics else [])
    opt = torch.optim.Adam(params, lr=1e-3)
    for it in range(n_iter):
        loss = torch.mean((net(t_d) - x_d)**2)                 # data term
        if use_physics:                                        # ODE residual term
            x = net(t_c)
            dx = torch.autograd.grad(x.sum(), t_c, create_graph=True)[0]
            ddx = torch.autograd.grad(dx.sum(), t_c, create_graph=True)[0]
            res = ddx + 2 * torch.exp(log_g) * dx + torch.exp(log_w2) * x
            loss = loss + lam * torch.mean(res**2)
        opt.zero_grad()
        loss.backward()
        opt.step()
    return net, np.exp(log_g.item()), np.sqrt(np.exp(log_w2.item()))

t_test = np.linspace(0, 6, 601)
x_true = damped_oscillator(t_test, 1.0, 0.3, 6.0, 0.5)
for use_physics in (False, True):
    net, g, w0 = train(use_physics)
    with torch.no_grad():
        x_pred = net(torch.tensor(t_test, dtype=torch.float32)[:, None]).numpy().ravel()
    err = x_pred - x_true
    print(f"physics={use_physics}: RMSE t<=3 {np.sqrt(np.mean(err[t_test<=3]**2)):.3f}, "
          f"t>3 {np.sqrt(np.mean(err[t_test>3]**2)):.3f}")
    if use_physics:
        print(f"  gamma = {g:.3f} (true 0.3), omega_0 = {w0:.3f} (true 6.007)")

**ส่วนตรวจผล** (ไม่อยู่ในตำรา): พิมพ์ตัวเลขที่ตำรายกมาเพื่อเทียบ

In [ ]:
# --- end of listings ---------------------------------------------------------

if __name__ == '__main__':
    print(f"both trainings: {time.time() - _t0:.0f} s")
    net_d, _, _ = train(False, n_iter=0)
    print(f"parameters of the network: {sum(q.numel() for q in net_d.parameters())}")

    from scipy.optimize import curve_fit
    t1 = time.time()
    p, c = curve_fit(damped_oscillator, t_obs, x_obs, p0=[1.0, 0.2, 6.0, 0.0])
    dt = time.time() - t1
    e = np.sqrt(np.diag(c))
    err = damped_oscillator(t_test, *p) - x_true
    print(f"curve_fit ({dt:.3f} s): gamma = {p[1]:.3f} +/- {e[1]:.3f}, omega = {p[2]:.3f} +/- {e[2]:.3f}; "
          f"RMSE t<=3 {np.sqrt(np.mean(err[t_test <= 3]**2)):.3f}, t>3 {np.sqrt(np.mean(err[t_test > 3]**2)):.3f}")
    print(f"size of the ODE term relative to x: omega_0^4 = {6.007**4:.0f}")

    # the PINN (last net trained) over the prediction range, split in two halves
    with torch.no_grad():
        x_pinn = net(torch.tensor(t_test, dtype=torch.float32)[:, None]).numpy().ravel()
    for a, b in [(3, 4.5), (4.5, 6)]:
        k = (t_test > a) & (t_test <= b)
        print(f"PINN, {a} < t <= {b} s: RMSE {np.sqrt(np.mean((x_pinn[k] - x_true[k])**2)):.3f} m, "
              f"max |x| {np.abs(x_pinn[k]).max():.2f} m (truth {np.abs(x_true[k]).max():.2f} m)")

    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(8, 3.5))
    ax.plot(t_test, x_true, 'k', lw=1, label='truth')
    ax.plot(t_obs, x_obs, 'o', ms=4, label='30 data points')
    with torch.no_grad():
        ax.plot(t_test, net(torch.tensor(t_test, dtype=torch.float32)[:, None]).numpy().ravel(), label='PINN')
    ax.plot(t_test, damped_oscillator(t_test, *p), '--', label='curve_fit')
    ax.axvspan(3, 6, color='grey', alpha=0.15); ax.set_xlabel('t (s)'); ax.set_ylabel('x (m)'); ax.legend(fontsize=8)
    fig.tight_layout(); fig.savefig('outputs/ch10_pinn.png', dpi=100)
    print("saved outputs/ch10_pinn.png")

In [ ]:
# แสดงรูปที่ส่วนนี้สร้าง (สคริปต์บันทึกรูปลงไฟล์ใน outputs/)
from IPython.display import Image, display
import glob, os
for _f in sorted(glob.glob('outputs/**/*.png', recursive=True)):
    if os.path.getmtime(_f) >= float(os.environ['NB_T0']):
        print(_f); display(Image(_f))


## `07_mc_dropout_ensemble.py` — โค้ดที่ 10.8

Ch.10  Uncertainty of the polymer classifier: MC dropout, a 5-network deep ensemble, and configuration spread.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 07_mc_dropout_ensemble.py  จากโฟลเดอร์ของบท
__file__ = '07_mc_dropout_ensemble.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.10  Uncertainty of the polymer classifier: MC dropout, a 5-network deep ensemble, and configuration spread.

Book references: ssec:mc-dropout, ssec:deep-ensemble, ssec:dl-uq-PI, code:mc-dropout (line for line), tab:nn-uq
Continues from code:polymer-nn (model, X_all, P, train_phase_classifier, crossing), run here from 04_polymer_classifier.py.
Numbers quoted in the text:
  tab:nn-uq (T, mean P, dropout SD, ensemble SD, config SD)
    1.0: 1.00 0.001 0.002 0.00   1.4: 0.40 0.060 0.084 0.46   1.8: 0.25 0.044 0.052 0.40
    2.4: 0.05 0.011 0.016 0.20   3.0: 0.01 0.002 0.006 0.07
  crossings of the five members 1.33-1.40 (training spread ~0.03); report 1.36 +/- 0.03;
  config-to-config spread ~5x the prediction uncertainty; ensemble SD above dropout SD at every T
About 1 minute.
"""
import os
import sys
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn

HERE = Path(__file__).resolve().parent
sys.path.insert(0, str(HERE))
from _shared import chain_data, listing

os.chdir(HERE)
Path('outputs').mkdir(exist_ok=True)
_nb = chain_data({'__name__': 'polymer', 'np': np, 'torch': torch, 'nn': nn})
listing(_nb, '04_polymer_classifier.py', 'polymer-nn')        # prints accuracy, T*, mean P, spread
globals().update({k: _nb[k] for k in ('T_lab', 'T_grid', 'dist', 'model', 'X_all', 'P',
                                      'train_phase_classifier', 'crossing')})

### โค้ดที่ 10.8

In [ ]:
# --- code:mc-dropout ---------------------------------------------------------
def mc_dropout_predict(model, X, n_samples=50):
    model.train()                                   # keep dropout active
    with torch.no_grad():
        S = torch.stack([torch.softmax(model(X), dim=1)[:, 1]
                         for _ in range(n_samples)])
    model.eval()
    return S.mean(dim=0).numpy(), S.std(dim=0).numpy()

torch.manual_seed(1)
p_mc, s_mc = mc_dropout_predict(model, X_all)

ensemble = [train_phase_classifier(dist, T_lab, T_low=1.0, T_high=3.0, seed=s)
            for s in range(5)]                      # 5 independently initialised nets
P_ens = np.array([member[2] for member in ensemble])
T_stars = [crossing(p, T_lab, T_grid)[0] for p in P_ens]
print("T* of members:", np.round(T_stars, 2))
for T in T_grid:
    m = T_lab == T
    print(f"T = {T:.1f}  MC-dropout sd {s_mc[m].mean():.3f}  "
          f"ensemble sd {P_ens[:, m].std(axis=0).mean():.3f}  "
          f"config-to-config sd {P[m].std():.2f}")

**ส่วนตรวจผล** (ไม่อยู่ในตำรา): พิมพ์ตัวเลขที่ตำรายกมาเพื่อเทียบ

In [ ]:
# --- end of listings ---------------------------------------------------------

if __name__ == '__main__':
    Ts = np.array(T_stars)
    print(f"member crossings {Ts.min():.2f}-{Ts.max():.2f}; mean {Ts.mean():.2f} +/- {Ts.std(ddof=1):.2f} (SD of members)")
    sd_mc = np.array([s_mc[T_lab == t].mean() for t in T_grid])
    sd_en = np.array([P_ens[:, T_lab == t].std(axis=0).mean() for t in T_grid])
    sd_cf = np.array([P[T_lab == t].std() for t in T_grid])
    pbar = np.array([P[T_lab == t].mean() for t in T_grid])
    mid = (T_grid >= 1.2) & (T_grid <= 2.0)
    print(f"ensemble SD > dropout SD at {np.sum(sd_en > sd_mc)}/{len(T_grid)} temperatures; "
          f"exceptions: {[(float(t), round(float(a), 5), round(float(b), 5)) for t, a, b in zip(T_grid, sd_en, sd_mc) if a <= b]}")
    print(f"T = 1.2-2.0: config SD / ensemble SD = {(sd_cf[mid] / sd_en[mid]).min():.1f}-{(sd_cf[mid] / sd_en[mid]).max():.1f}, "
          f"config SD / dropout SD = {(sd_cf[mid] / sd_mc[mid]).min():.1f}-{(sd_cf[mid] / sd_mc[mid]).max():.1f}")
    print("\ntab:nn-uq")
    for t in (1.0, 1.4, 1.8, 2.4, 3.0):
        k = np.isclose(T_grid, t)
        print(f"  {t:.1f} & {pbar[k][0]:.2f} & {sd_mc[k][0]:.3f} & {sd_en[k][0]:.3f} & {sd_cf[k][0]:.2f}")

    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(6.5, 3.8))
    pm_mc = np.array([p_mc[T_lab == t].mean() for t in T_grid])
    pm_en = P_ens.mean(0); pm_en = np.array([pm_en[T_lab == t].mean() for t in T_grid])
    ax.fill_between(T_grid, pm_mc - sd_mc, pm_mc + sd_mc, alpha=0.3, label='MC dropout (mean SD)')
    ax.fill_between(T_grid, pm_en - sd_en, pm_en + sd_en, alpha=0.3, label='ensemble of 5 (mean SD)')
    ax.errorbar(T_grid, pbar, yerr=sd_cf, fmt='ko', ms=3, capsize=2, label='config-to-config SD')
    ax.axhline(0.5, color='grey', lw=0.5); ax.set_xlabel('T'); ax.set_ylabel(r'$\bar P$(globule)'); ax.legend(fontsize=8)
    fig.tight_layout(); fig.savefig('outputs/ch10_uncertainty.png', dpi=100)
    print("saved outputs/ch10_uncertainty.png")

In [ ]:
# แสดงรูปที่ส่วนนี้สร้าง (สคริปต์บันทึกรูปลงไฟล์ใน outputs/)
from IPython.display import Image, display
import glob, os
for _f in sorted(glob.glob('outputs/**/*.png', recursive=True)):
    if os.path.getmtime(_f) >= float(os.environ['NB_T0']):
        print(_f); display(Image(_f))


## `09_replicate_chains.py` — โค้ดที่ 10.3, 10.6

Ch.10  How much do the polymer results depend on the particular simulated ensemble?

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 09_replicate_chains.py  จากโฟลเดอร์ของบท
__file__ = '09_replicate_chains.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.10  How much do the polymer results depend on the particular simulated ensemble?

The chain of code:chain-gen moves in continuous space, so its Metropolis trajectory is chaotic with respect to
floating-point rounding: a different CPU or numpy build (e.g. Apple Silicon vs Linux x86) flips one accept/reject
decision sooner or later and from then on produces a different - but statistically equivalent - ensemble.
(The Ising configurations of code:ising-configs are bit-identical everywhere: spins are integers.)
This script repeats code:chain-gen with four other seed sets (100 + k + 1000 r, r = 1..4; r = 0 is the book's run)
and reports the quantities the text quotes, to show the run-to-run spread a reader should expect.
About 10 minutes on two cores (two simulations at a time).
"""
import os
import sys
from multiprocessing import Pool
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn

HERE = Path(__file__).resolve().parent
sys.path.insert(0, str(HERE))
from _shared import section, chain_data

os.chdir(HERE)
GEN = section(HERE / '03_polymer_chain.py', '\n# --- code:chain-gen', '\nT_grid = ')
T_grid = np.round(np.arange(0.6, 3.41, 0.2), 1)


def simulate(job):
    r, k, T = job
    g = {'np': np}
    exec(GEN, g)
    return g['make_chain_ensemble'](T=T, seed=100 + k + 1000 * r)


def analyse(conf, ener, T_lab):
    from scipy.optimize import curve_fit
    torch.set_num_threads(1)
    nb = {'np': np, 'torch': torch, 'nn': nn}
    exec(section(HERE / '04_polymer_classifier.py', '\n# --- code:polymer-nn', 'model, X_all, P, acc ='), nb)
    iu, ju = np.triu_indices(30, k=1)
    dist = np.linalg.norm(conf[:, iu] - conf[:, ju], axis=2)
    rg2 = np.mean(np.sum(conf**2, axis=2), axis=1)
    mean = np.array([rg2[T_lab == T].mean() for T in T_grid])
    f = lambda T, a, b, Ts, w: a + b * np.tanh((T - Ts) / w)
    p, c = curve_fit(f, T_grid, mean, p0=[5, 3, 1.8, 0.5], maxfev=20000)
    ts = [nb['crossing'](nb['train_phase_classifier'](dist, T_lab, 1.0, 3.0, seed=s)[2], T_lab, T_grid)[0] for s in range(5)]
    _, _, P, acc = nb['train_phase_classifier'](dist, T_lab, 1.0, 3.0)
    Pm = nb['crossing'](P, T_lab, T_grid)[1]
    cv06 = ener[T_lab == 0.6].var() / 0.36 / 30
    return p[2], np.sqrt(c[2, 2]), p[3], np.array(ts), Pm, cv06, mean


if __name__ == '__main__':
    rows = []
    book = chain_data()
    rows.append((0,) + analyse(book['conf'], book['ener'], book['T_lab']))
    jobs = [(r, k, T) for r in (1, 2, 3, 4) for k, T in enumerate(T_grid)]
    with Pool(2) as pool:
        res = pool.map(simulate, jobs)
    T_lab = np.repeat(T_grid, 400)
    for r in (1, 2, 3, 4):
        d = [res[i] for i, j in enumerate(jobs) if j[0] == r]
        conf = np.concatenate([x[0] for x in d]); ener = np.concatenate([x[1] for x in d])
        rows.append((r,) + analyse(conf, ener, T_lab))
    print(" r  tanh T*        w     T*_NN seeds 0-4 (mean)            P(1.2) P(1.4) P(1.6)  Cv/N(0.6)  <Rg2>(0.6, 3.4)")
    for r, Ts, eTs, w, ts, Pm, cv, mean in rows:
        print(f"{r:2d}  {Ts:.2f} +/- {eTs:.2f}  {w:.2f}  {np.round(ts, 2)} ({ts.mean():.2f})  "
              f"{Pm[3]:.2f}   {Pm[4]:.2f}   {Pm[5]:.2f}    {cv:.2f}      {mean[0]:.2f}, {mean[-1]:.2f}")
    allT = np.array([x[1] for x in rows]); allNN = np.array([x[4].mean() for x in rows])
    print(f"across the five ensembles: tanh T* {allT.min():.2f}-{allT.max():.2f} (SD {allT.std(ddof=1):.2f}); "
          f"T*_NN {allNN.min():.2f}-{allNN.max():.2f} (SD {allNN.std(ddof=1):.2f}); "
          f"gap tanh - NN {np.min(allT - allNN):.2f}-{np.max(allT - allNN):.2f}")

# แนวคำตอบของแบบฝึกหัดที่ต้องรันโค้ด
ควรลองทำเองก่อน แล้วจึงรันเซลล์เหล่านี้เพื่อเทียบคำตอบ

## `exercises/E10_2_width_activation.py` — โค้ดที่ 10.1, 10.2

E10.2  Network width and the role of the non-linearity (Higgs MLP, code:mlp + code:train).

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E10_2_width_activation.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E10_2_width_activation.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E10.2  Network width and the role of the non-linearity (Higgs MLP, code:mlp + code:train).

Five networks: ReLU with hidden width 4, 16, 64, 256, and width 64 with tanh; same seed, same loop.
Answer in the text: AUC ~0.971 / 0.978 / 0.979 / 0.979, tanh 0.978; accuracy 0.920-0.935;
parameters 50 -> ~67k; width 4 starts highest (~0.60) and ends at 0.160, the others end at 0.123-0.136.
About 1.5 minutes.
"""
import os
import sys
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, TensorDataset

HERE = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(HERE))
from _shared import higgs_split, listing

os.chdir(HERE)
nb = higgs_split({})
listing(nb, '01_higgs_mlp.py', 'mlp', '\n# --- code:train')
Xt, yt, Xv, y_val = nb['Xt'], nb['yt'], nb['Xv'], nb['y_val']


def make(hidden, act):
    return nn.Sequential(nn.Linear(4, hidden), act(), nn.Linear(hidden, hidden), act(), nn.Linear(hidden, 2))


print(f"{'network':12s} {'params':>7s}  loss ep1  loss ep30   AUC    acc")
for name, hidden, act in [('ReLU 4', 4, nn.ReLU), ('ReLU 16', 16, nn.ReLU), ('ReLU 64', 64, nn.ReLU),
                          ('ReLU 256', 256, nn.ReLU), ('tanh 64', 64, nn.Tanh)]:
    torch.manual_seed(0)
    model = make(hidden, act)
    loader = DataLoader(TensorDataset(Xt, yt), batch_size=64, shuffle=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()
    history = []
    for epoch in range(30):
        model.train(); total = 0.0
        for xb, yb in loader:
            loss = loss_fn(model(xb), yb)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            total += loss.item() * len(xb)
        history.append(total / len(Xt))
    model.eval()
    with torch.no_grad():
        p = torch.softmax(model(Xv), dim=1)[:, 1].numpy()
    n_par = sum(q.numel() for q in model.parameters())
    print(f"{name:12s} {n_par:7d}  {history[0]:8.3f}  {history[-1]:9.3f}  {roc_auc_score(y_val, p):.3f}  {np.mean((p > 0.5) == y_val):.3f}")

## `exercises/E10_3_training_bugs.py` — โค้ดที่ 10.2

E10.3  Three bugs in the training loop of code:train, one at a time, same seed.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E10_3_training_bugs.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E10_3_training_bugs.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E10.3  Three bugs in the training loop of code:train, one at a time, same seed.

  bug 1  optimizer.zero_grad() removed      -> gradients accumulate over all previous steps
  bug 2  torch.softmax applied before CrossEntropyLoss (softmax twice)
  bug 3  no StandardScaler (raw GeV features)
Answer in the text: bug 1 AUC ~0.873, accuracy 0.797 (< 0.800 of 'always background');
bug 2 AUC 0.978 but final loss ~0.366 instead of 0.126; bug 3 AUC ~0.945.
About 1 minute.
"""
import os
import sys
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, TensorDataset

HERE = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(HERE))
from _shared import higgs_split, listing

os.chdir(HERE)
nb = higgs_split({})
listing(nb, '01_higgs_mlp.py', 'mlp', '\n# --- code:train')
HiggsMLP, y_train, y_val = nb['HiggsMLP'], nb['y_train'], nb['y_val']


def run(zero_grad=True, softmax_twice=False, scale=True):
    if scale:
        Xt, Xv = nb['Xt'], nb['Xv']
    else:
        Xt = torch.tensor(nb['X_train'], dtype=torch.float32)
        Xv = torch.tensor(nb['X_val'], dtype=torch.float32)
    yt = torch.tensor(y_train)
    torch.manual_seed(0)
    model = HiggsMLP()
    loader = DataLoader(TensorDataset(Xt, yt), batch_size=64, shuffle=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()
    history = []
    for epoch in range(30):
        model.train(); total = 0.0
        for xb, yb in loader:
            out = model(xb)
            if softmax_twice:
                out = torch.softmax(out, dim=1)                # BUG 2
            loss = loss_fn(out, yb)
            if zero_grad:
                optimizer.zero_grad()                          # BUG 1 when skipped
            loss.backward(); optimizer.step()
            total += loss.item() * len(xb)
        history.append(total / len(Xt))
    model.eval()
    with torch.no_grad():
        p = torch.softmax(model(Xv), dim=1)[:, 1].numpy()
    return history, p


print(f"accuracy of 'always background': {1 - y_val.mean():.3f}")
print(f"{'case':22s} loss ep1  loss ep30   AUC    acc")
for name, kw in [('correct loop', {}), ('no zero_grad', {'zero_grad': False}),
                 ('softmax twice', {'softmax_twice': True}), ('no scaling', {'scale': False})]:
    h, p = run(**kw)
    print(f"{name:22s} {h[0]:8.3f}  {h[-1]:9.3f}  {roc_auc_score(y_val, p):.3f}  {np.mean((p > 0.5) == y_val):.3f}")

## `exercises/E10_4_ising_extremes.py` — โค้ดที่ 10.5, 10.6

E10.4  Test learning by extremes on a system with a known answer: the 16x16 Ising model (T_c = 2.269).

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E10_4_ising_extremes.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E10_4_ising_extremes.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E10.4  Test learning by extremes on a system with a known answer: the 16x16 Ising model (T_c = 2.269).

train_phase_classifier and crossing from code:polymer-nn, applied to the 256 raw spins of code:ising-configs
(data/ising.npz from 05_ising_configs.py, generated here if missing, about one minute).
Answer in the text: ends (1.8, 2.8) -> 2.39-2.41; ends (2.0, 2.6) -> 2.35-2.36; <|m|> ~ 0.58 at T = 2.4;
ends (1.6, 3.2) -> 1.65 with mean P = 0 at T = 1.7, because every configuration at T = 1.6 has m < 0 and
every one at T = 1.7 has m > 0; with spin-flip augmentation (s -> -s) the crossing returns to 2.43-2.45.
About 2 minutes.
"""
import os
import subprocess
import sys
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn

HERE = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(HERE))
from _shared import listing

os.chdir(HERE)
if not Path('data/ising.npz').exists():
    subprocess.run([sys.executable, '05_ising_configs.py'], check=True)
d = np.load('data/ising.npz')
spins, T_spin = d['spins'], d['T_spin']
T_ising = np.unique(T_spin)

nb = {'np': np, 'torch': torch, 'nn': nn}
listing(nb, '04_polymer_classifier.py', 'polymer-nn', 'model, X_all, P, acc =')   # definitions only
train_phase_classifier, crossing = nb['train_phase_classifier'], nb['crossing']

m = spins.mean(axis=1)
print(f"<|m|> at T = 2.4: {np.abs(m[T_spin == 2.4]).mean():.2f}")
print("fraction of configurations with m > 0:",
      ' '.join(f"{t:.1f}:{np.mean(m[T_spin == t] > 0):.2f}" for t in T_ising))

for lo, hi in [(1.8, 2.8), (2.0, 2.6), (1.6, 3.2)]:
    res = []
    for seed in range(3):
        _, _, P, acc = train_phase_classifier(spins, T_spin, lo, hi, seed=seed)
        ts, Pm = crossing(P, T_spin, T_ising)
        res.append(ts)
        if seed == 0:
            first = Pm
    print(f"ends <= {lo}, >= {hi}: crossings {np.round(res, 3)}  (T_c = 2.269)")
    if lo == 1.6:
        print("   mean P by T (seed 0):", ' '.join(f"{t:.1f}:{p:.2f}" for t, p in zip(T_ising, first)))

# symmetry s -> -s: add every configuration with all spins flipped
S2, T2 = np.concatenate([spins, -spins]), np.concatenate([T_spin, T_spin])
res = [crossing(train_phase_classifier(S2, T2, 1.6, 3.2, seed=seed)[2], T2, T_ising)[0] for seed in range(3)]
print(f"ends <= 1.6, >= 3.2 with spin-flip augmentation: crossings {np.round(res, 3)}")

## `exercises/E10_6_dropout_rate_ends.py` — โค้ดที่ 10.5, 10.8

E10.6  How much of the network uncertainty is a choice? Dropout rate and training ends.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E10_6_dropout_rate_ends.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E10_6_dropout_rate_ends.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E10.6  How much of the network uncertainty is a choice? Dropout rate and training ends.

1. Retrain the classifier of code:polymer-nn with dropout 0.05 and 0.5 (instead of 0.2) and repeat MC dropout
   (code:mc-dropout): the dropout SD changes with the rate while the crossing stays put.
2. Train the 5-network ensemble with ends (T <= 0.8, T >= 3.2) and compare the crossings with the
   original ensemble (ends 1.0 / 3.0: 1.33-1.40).
Answer in the text: maximum dropout SD ~0.07 (rate 0.2), ~0.04 (0.05), ~0.14 (0.5); crossing still 1.33;
ensemble with ends 0.8 / 3.2: crossings 1.11-1.25, no overlap with 1.33-1.40, wider spread.
About 1.5 minutes.
"""
import os
import sys
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn

HERE = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(HERE))
from _shared import chain_data, section

os.chdir(HERE)
nb = chain_data({'np': np, 'torch': torch, 'nn': nn})
T_lab, T_grid, dist = nb['T_lab'], nb['T_grid'], nb['dist']
defs = section(HERE / '04_polymer_classifier.py', '\n# --- code:polymer-nn', 'model, X_all, P, acc =')
exec(section(HERE / '07_mc_dropout_ensemble.py', '\n# --- code:mc-dropout', 'torch.manual_seed(1)'), nb)   # mc_dropout_predict

for rate in (0.05, 0.2, 0.5):
    exec(defs.replace('nn.Dropout(0.2)', f'nn.Dropout({rate})'), nb)
    model, X_all, P, acc = nb['train_phase_classifier'](dist, T_lab, T_low=1.0, T_high=3.0)
    torch.manual_seed(1)
    p_mc, s_mc = nb['mc_dropout_predict'](model, X_all)
    sd = np.array([s_mc[T_lab == t].mean() for t in T_grid])
    print(f"dropout {rate:4.2f}: val. acc {acc:.3f}, crossing {nb['crossing'](P, T_lab, T_grid)[0]:.2f}, "
          f"max MC-dropout SD {sd.max():.3f} at T = {T_grid[sd.argmax()]:.1f}")

exec(defs, nb)                                                 # back to dropout 0.2
for lo, hi in [(1.0, 3.0), (0.8, 3.2)]:
    ts = [nb['crossing'](nb['train_phase_classifier'](dist, T_lab, lo, hi, seed=s)[2], T_lab, T_grid)[0] for s in range(5)]
    print(f"ensemble with ends <= {lo}, >= {hi}: crossings {np.round(ts, 2)}, range {np.ptp(ts):.2f}, SD {np.std(ts, ddof=1):.3f}")